# ComfyUI (no custom nodes)

Installs ComfyUI with only the local `model-manager` custom node. Needs a GPU runtime and an ephemeral Tailscale auth key stored as the `TS_AUTHKEY` Colab secret.
The Models sidebar tab installs the models listed in `colab/models.json` using the `CIVITAI_API_KEY` and `HUG_TOKEN` Colab secrets.

In [ ]:
!git clone https://github.com/trailbat/ComfyUI.git /content/ComfyUI
# aria2 and aria2p are used by colab/downloader.py.
!apt-get update -qq && apt-get install -y -qq aria2
# Colab ships a CUDA build of torch, the unpinned torch requirement keeps it.
!cd /content/ComfyUI && uv pip install --system -r requirements.txt -r manager_requirements.txt aria2p

In [ ]:
import os
from google.colab import userdata
os.environ["TS_AUTHKEY"] = userdata.get("TS_AUTHKEY")
!python /content/ComfyUI/colab/tailscale.py

In [ ]:
import sys
sys.path.append("/content/ComfyUI/colab")
from downloader import Downloader

with Downloader("/content/ComfyUI/models") as downloader:
    pass  # e.g. downloader.lora("https://civitai.com/api/download/models/...", rename="name.safetensors")

In [ ]:
import os
from google.colab import userdata
# Colab secrets can't be read from the ComfyUI process, the Models sidebar reads them from the environment.
for secret in ["CIVITAI_API_KEY", "HUG_TOKEN"]:
    os.environ[secret] = userdata.get(secret)
!python /content/ComfyUI/main.py --enable-manager --listen